# IE-OPT-06 · 单周期随机需求库存优化（原创 PySCIPOpt 教学案例）

假定一个备件只能在需求实现前订购一次。可能的需求及其概率已给定。目标是优化订货量，最小化期望持有与缺货成本。

学习参考：[PySCIPOpt 官方 EOQ 案例 eoq_en.py](https://github.com/scipopt/PySCIPOpt/blob/master/examples/finished/eoq_en.py)。

**严谨区分**：官方 EOQ 为有容量约束的多品种确定性补货近似；本站使用一周期离散需求场景的随机库存模型。另有 [IE-MC-02](../inventory-lab.html) 处理多日随机需求与补货提前期仿真，三者不是同一数学模型。数据为合成教学场景。

## 1. 两阶段场景法

- 决策：在不知道实际需求 d_s 时确定整数订货量 q。
- 场景 s 有概率 p_s；需求实现后形成剩余库存 h_s=max(q-d_s,0)、缺口 u_s=max(d_s-q,0)。
- 线性化：h_s≥q-d_s、u_s≥d_s-q 且均非负。
- 最小化 Σ_s p_s [c_h h_s+c_u u_s]。由于两项成本均为正，最优解会将松弛量取到正部。
- 这里未计采购成本、固定订货费、提前期与收入。

In [ ]:
from pyscipopt import Model, quicksum

demands = [1, 2, 3, 4, 5]
probabilities = [0.10, 0.20, 0.40, 0.20, 0.10]  # 合成场景
holding_cost = 1.0
shortage_cost = 6.0
max_order = 10
assert abs(sum(probabilities) - 1) < 1e-10

In [ ]:
def solve_newsvendor(demands, probabilities, holding_cost, shortage_cost, max_order):
    n = len(demands)
    model = Model("IE-OPT-06 discrete stochastic inventory")
    model.hideOutput(True)
    q = model.addVar(vtype="I", lb=0, ub=max_order, name="order_quantity")
    leftover = [model.addVar(lb=0, name=f"leftover_{i}") for i in range(n)]
    shortage = [model.addVar(lb=0, name=f"shortage_{i}") for i in range(n)]
    for i in range(n):
        model.addCons(leftover[i] >= q - demands[i])
        model.addCons(shortage[i] >= demands[i] - q)
    model.setObjective(quicksum(probabilities[i] *
        (holding_cost * leftover[i] + shortage_cost * shortage[i])
        for i in range(n)), "minimize")
    model.optimize()
    assert str(model.getStatus()) == "optimal"
    return {"q": int(round(model.getVal(q))),
            "expected_cost": float(model.getObjVal())}

answer = solve_newsvendor(demands, probabilities, holding_cost, shortage_cost, max_order)
print("最优订货量:", answer["q"])
print("最小期望损失:", answer["expected_cost"])

## 2. 穷举 0～10 件验证最优值

独立计算每个整数订货量在全部需求场景中的期望成本。若 MIP 模型、概率加权或正部约束有错，穷举对照会失败。

In [ ]:
def explicit_expected_cost(q, demands, probs, holding, shortage):
    return sum(probs[i] * (holding * max(0, q - demands[i]) +
                           shortage * max(0, demands[i] - q))
               for i in range(len(demands)))

values = [explicit_expected_cost(q, demands, probabilities,
          holding_cost, shortage_cost) for q in range(max_order + 1)]
oracle = min(values)
assert abs(answer["expected_cost"] - oracle) < 1e-7
assert abs(explicit_expected_cost(answer["q"], demands, probabilities,
           holding_cost, shortage_cost) - oracle) < 1e-7
print("独立穷举校验通过")
print("候选订货量与期望损失:", list(enumerate(round(x, 3) for x in values)))

## 3. 课堂讨论

1. 将缺货损失从 6 提升到 12，最优订货量如何变化？结合成本不对称解释。
2. 将概率分布改成更重尾的需求场景，原订货量是否仍合理？
3. 概率若来自样本估计，为什么需要估计误差与样本外测试？切勿把测试样本用作事前已知信息。
4. 与多天 (R,S) 库存仿真比较，哪些状态和补货时点在本模型中根本不存在？

**这是未接入 L2 正式成绩的课堂 Notebook；仿真与优化结论只能在相应假设内解释。**